In [ ]:
import SimpleITK as sitk

def resample_volume(input_image, new_spacing=[1.0, 1.0, 1.0], interpolator=sitk.sitkNearestNeighbor, new_size=None):
    """
    Resample the given 3D volume to new spacing and size.
    
    Parameters:
        input_image (SimpleITK.Image): The input image.
        new_spacing (list of float): The new spacing (voxel size) in each dimension.
        interpolator (SimpleITK.InterpolatorEnum): The interpolation method to use.
        new_size (list of int): New number of pixels/voxels in each dimension. If None, calculate based on original size and new spacing.
    
    Returns:
        SimpleITK.Image: The resampled image.
    """
    # Original size and spacing
    original_size = input_image.GetSize()
    original_spacing = input_image.GetSpacing()
    
    # If new size is not provided, compute it based on the original size and spacing ratios
    if new_size is None:
        new_size = [
            int(round(osz * ospc / nspc)) for osz, ospc, nspc in zip(original_size, original_spacing, new_spacing)
        ]
    
    # Transformation setup - identity, as we are just resizing
    transform = sitk.Transform(3, sitk.sitkIdentity)
    
    # Setup the resample algorithm
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputOrigin(input_image.GetOrigin())
    resampler.SetOutputDirection(input_image.GetDirection())
    resampler.SetTransform(transform)
    resampler.SetInterpolator(interpolator)
    
    # Execute resampling
    new_image = resampler.Execute(input_image)
    return new_image

# Example usage
if __name__ == "__main__":
    # Load an image (modify path as necessary)
    input_img_path = 'path_to_your_image.nrrd'
    input_image = sitk.ReadImage(input_img_path)
    
    # Define new spacing: [x, y, z] in millimeters
    new_spacing = [0.8, 0.8, 0.8]  # Example: new voxel size of 2mm x 2mm x 2mm
    
    # Resample the image using nearest neighbor interpolation
    resampled_image = resample_volume(input_image, new_spacing=new_spacing, interpolator=sitk.sitkNearestNeighbor)
    
    # Save the resampled image
    output_img_path = 'path_to_your_resampled_image.nrrd'
    sitk.WriteImage(resampled_image, output_img_path)
    print("Resampling completed. Output saved to:", output_img_path)


In [21]:
# GOLD STANDARD TO DISPLAY MAYAVI VIEWERS WORKS WITH NRRD

from mayavi import mlab
import numpy as np
import nrrd
import SimpleITK as sitk

def update_camera_info(obj, evt):
    camera = mlab.gcf().scene.camera
    print("Camera Position:", camera.position)
    print("Camera Focal Point:", camera.focal_point)
    print("Camera View Up:", camera.view_up)

LabelPath ='Y:/DeepLearningMONAI/ROH-2024/Case_04_Inference.nrrd'
model_path= 'Y:/DeepLearningMONAI/ROH-2024/Case_04_Inference.nrrd'
avmlabel, metadata = nrrd.read(LabelPath)
OP = 1
DecimationFactor = 1
figure_name = "Case_01"                     ##LabelPath.split('/')[-4]  # Assuming '/' is the path separator

model = sitk.ReadImage(model_path)
array = sitk.GetArrayFromImage(model)
shape = array.shape
print(shape)

# array1= array[0][0][0]
# print(array1)
  
min_index_depth = 0
max_index_depth = shape[0]

min_index_height = 0
max_index_height = shape[1] 

min_index_width = 0
max_index_width = shape[2] 

# origin = array.GetOrigin()
# dimensions = array.GetSize()
# direction = array.GetDirection()


print("Minimum and Maximum Indices along each Axis:")
print(f"Depth (Axis 0): Min index = {min_index_depth}, Max index = {max_index_depth}")
print(f"Height (Axis 1): Min index = {min_index_height}, Max index = {max_index_height}")
print(f"Width (Axis 2): Min index = {min_index_width}, Max index = {max_index_width}")

spacing = (0.8, 0.8, 0.8)

green = (128/255, 174/255, 128/255)
yellow = (241/255, 214/255, 145/255)

x = (min_index_depth + max_index_depth) / 2
y = (min_index_height + max_index_height) / 2
z = (min_index_width + max_index_width) / 2

ypt = y + 600
# Camera Position: [223.00919646 635.03662316 226.60608859]
# Camera Focal Point: [193.63645894 122.20485721 134.50254464]
# Camera View Up: [-0.00882131 -0.17627335  0.98430173]

print(x/2)
print(y/2)
print(z/2)

# Create a figure with a specific name and size
fig = mlab.figure(figure_name, bgcolor=(1, 1, 1), size=(1400, 1400))
# obj = (avmlabel == 1).astype(int)
# mlab.contour3d(obj[::DecimationFactor,::DecimationFactor,::DecimationFactor], contours=[0.5], color=green, opacity=OP)
# obj = (avmlabel == 2).astype(int)
# mlab.contour3d(obj[::DecimationFactor,::DecimationFactor,::DecimationFactor], contours=[0.5], color=yellow, opacity=OP)

obj = (avmlabel == 1).astype(int)
mlab.contour3d(obj[:,:,:], color=green, opacity=OP)
obj = (avmlabel == 2).astype(int)
mlab.contour3d(obj[:,:,:], color=yellow, opacity=0.02)


camera = mlab.gcf().scene.camera
mlab.gcf().scene.camera.position = [x, ypt, z]
mlab.gcf().scene.camera.focal_point = [x, y, z] # 
mlab.gcf().scene.camera.view_up = [0, 0, 1] # good
mlab.gcf().scene.camera.clipping_range = (1,10000)
mlab.gcf().scene.interactor.add_observer('MouseMoveEvent', update_camera_info)

# mlab.savefig("Y:/DeepLearningMONAI/ROH-2024/Case_04_Inference_A.png")

mlab.show()

(303, 233, 330)
Minimum and Maximum Indices along each Axis:
Depth (Axis 0): Min index = 0, Max index = 303
Height (Axis 1): Min index = 0, Max index = 233
Width (Axis 2): Min index = 0, Max index = 330
75.75
58.25
82.5
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Point: [151.5 116.5 165. ]
Camera View Up: [0. 0. 1.]
Camera Position: [151.5 716.5 165. ]
Camera Focal Po